In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

## [RAG절차]
1. 문서를 읽는다
    * %pip install -u -q docx2txt
2. 문서를 쪼갠다
    * %pip install -qU langchain-text-splitters
3. 쪼갠 문서를 임베딩하여 vector database에 넣음(local에 저장) cf. 클라우드에 저장
    * %pip install -q langchain-chroma
4. 질문을 이용해 유사도 검색
5. 유사도 검색한 문서를 LLM에 질문과 함꼐 전달하여 답변을 얻음(렝체인 사용 가능)
    * %pip install langchain
    * (https://www.langchain.com 에서 key 생성 .env에 LANGCHAIN_API_KEY로 추가)

# 0. 패키지 설치

In [ ]:
# 문서 읽어오기
pip install -u -q docx2txt

In [2]:
# 텍스트를 chunk로 나누는 기능만 있는 경량 모듈
%pip install -qU langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 벡터DB(로컬DB) 어제의 chromadb가 아님
%pip install -q langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [4]:
# langchain 사용
%pip install langchain


   ---------------------- ----------------- 4/7 [langgraph-prebuilt]
   ---------------------------- ----------- 5/7 [langgraph]
   ---------------------------------------- 7/7 [langchain]

Note: you may need to restart the kernel to use updated packages.


# 1. 문서읽기(X)

In [6]:
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./data/소득세법(법률)(제21065호)(20260102).docx')
document = loader.load()

In [7]:
len(document)

1

# 2. 문서를 쪼개면서 (O)
- https://docs.langchain.com/oss/python/integrations/splitters
## 2.1 1500토큰씩 쪼개서 읽어오기

In [15]:
import time
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('./data/소득세법(법률)(제21065호)(20260102).docx')
# gpt-4, gpt-4o, gpt-4 turbo, gpt4o-mini. embedding 모델들은 다 같은 방식으로 토큰 추출
text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base", # 토큰을 세는 방식 이름
    chunk_size=1500,             # chunk당 토큰 수
    chunk_overlap=200
    # seperators = ['\n', '\n\n'] 파라미터가 없음
)
start = time.time()
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 쪼개면서 읽는 시간 :', runtime)

문서를 쪼개면서 읽는 시간 : 3.870635747909546


In [12]:
len(document)

180

In [17]:
# chunk 글자수
# documents[0].page_content
print([len(document.page_content) for document in documents])

[1699, 1656, 1641, 1650, 1738, 1442, 1287, 1535, 1325, 1619, 1596, 1588, 1566, 1639, 1622, 1559, 1612, 1638, 1573, 1465, 1436, 1609, 1456, 1497, 1635, 1606, 1533, 1649, 1662, 1595, 1603, 1678, 1595, 1637, 1601, 1539, 1561, 1594, 1693, 1708, 1657, 1627, 1636, 1659, 1667, 1595, 1491, 1485, 1645, 1709, 1629, 1617, 1495, 1626, 1612, 1620, 1609, 1576, 1636, 1602, 1556, 1563, 1600, 1616, 1643, 1691, 1635, 1685, 1621, 1631, 1609, 1605, 1603, 1604, 1698, 1686, 1702, 1612, 1539, 1558, 1651, 2060, 1562, 1606, 1557, 1648, 1594, 1615, 1766, 1651, 1690, 1576, 1536, 1553, 1638, 1685, 1693, 1694, 1664, 1529, 1627, 1703, 1675, 1546, 1585, 1687, 1679, 1714, 1603, 1655, 1648, 1495, 1531, 1562, 1594, 1646, 1543, 1449, 1593, 1559, 1521, 1473, 1519, 1545, 1668, 1700, 1692, 1655, 1648, 1741, 1670, 1628, 1639, 1623, 1638, 1642, 1666, 1658, 1594, 1591, 1561, 1641, 1498, 1610, 1567, 1613, 1636, 1619, 1531, 1496, 1702, 1598, 1579, 1627, 1559, 1585, 1665, 1565, 1616, 1564, 1612, 1535, 1512, 1557, 1576, 1628, 165

In [19]:
# chunk 글자수 최댓값, 최솟값
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents[:-1]]))

2060
1287


## 2.2 1500 글자 쪼개서 읽어오기

In [1]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문자를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=200,
    # separators=["\n\n", "\n", " ",""] # 기본값
)
# 재귀적으로 다음 순서대로 시도:
# 1. \n\n(문단구분)
# 2. \n(줄바꿈)
# 3. " "(공백)
# 4. "" - 최후에는 글자 단위로 chunking
documents=loader.load_and_split(text_splitter=text_splitter)
runtime = time.time()-start
print('문서를 1500글자 즈음으로 쪼개면서 읽는 시간 :', runtime)
print("chunk 갯수 :", len(documents))

문서를 1500글자 즈음으로 쪼개면서 읽는 시간 : 4.720542669296265
chunk 갯수 : 193


In [5]:
# chunk들의 글자수
print([len(document.page_content) for document in documents])

[1463, 1421, 1482, 1487, 1479, 1408, 1457, 1495, 1467, 1446, 1487, 1456, 1467, 1351, 1392, 1362, 1402, 1470, 1410, 1489, 1455, 1496, 1441, 1319, 1458, 1476, 1452, 1382, 1384, 1467, 1227, 1494, 1494, 1470, 1454, 1495, 1412, 1477, 1477, 1362, 1449, 1386, 1055, 1467, 1361, 1493, 1467, 1434, 1351, 1471, 1495, 1479, 1457, 1442, 1370, 873, 1419, 1357, 1353, 1316, 1349, 1452, 1439, 1363, 1433, 1412, 1306, 1200, 1411, 1452, 1421, 1318, 1416, 1333, 1308, 1385, 1479, 1495, 1399, 1375, 1360, 1353, 1382, 1446, 1356, 1409, 1483, 1486, 1157, 1233, 1443, 1474, 1369, 1439, 1451, 1495, 1443, 1489, 1484, 1407, 1432, 1436, 1468, 1442, 1477, 1396, 1423, 1282, 1496, 1486, 1376, 1342, 1466, 1385, 1491, 1477, 1470, 1385, 1477, 1445, 1485, 1373, 1495, 1443, 1419, 1456, 1451, 1305, 1454, 1411, 1443, 1488, 1404, 1419, 1339, 1451, 1288, 1450, 1481, 1419, 1369, 1479, 1480, 1461, 1414, 1419, 1463, 1481, 1486, 1387, 1485, 1448, 1367, 1364, 1391, 1446, 1414, 1414, 1414, 1473, 1417, 1474, 1419, 1342, 1406, 1338, 1138

In [7]:
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents][:-1]))

1496
873


# 3. 쪼갠 문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : text-embedding-3-large (기본모델 : text-embedding-ada-002)
- 벡터데이터베이스(벡터 store) : chroma

In [10]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [22]:
# embed_query() 한 문자열을 임베딩 벡터로 전환한 list를 return
embedding.embed_query("소득세법은 다음과 같다")

[0.014609415084123611,
 -0.00840421486645937,
 0.0004996765055693686,
 0.023311901837587357,
 0.011843101121485233,
 0.024633649736642838,
 -0.018176967278122902,
 0.04655362293124199,
 -0.013123909942805767,
 0.015170865692198277,
 0.03986300155520439,
 -0.0037137623876333237,
 0.04561787098646164,
 -0.021370217204093933,
 0.01663297787308693,
 -0.019334958866238594,
 0.00612332159653306,
 -0.019346656277775764,
 0.0008582592708989978,
 -0.0017296775477007031,
 -0.009965750388801098,
 0.006977194920182228,
 0.023732990026474,
 -0.004611498676240444,
 -0.06409895420074463,
 -0.0018978203879669309,
 0.01019968744367361,
 0.023803170770406723,
 0.02203693985939026,
 -0.017252912744879723,
 0.010895652696490288,
 0.01736988127231598,
 0.03293844312429428,
 -0.04578162729740143,
 0.015007109381258488,
 0.015311229042708874,
 0.019767744466662407,
 0.02283232845366001,
 0.012106280773878098,
 0.012784700840711594,
 -0.025896914303302765,
 0.030224762856960297,
 -0.030528882518410683,
 0.050

In [24]:
embedding_vector = embedding.embed_documents( # 여러 문자열을 임베딩 벡터로 
    [
        "소득세법은 어쩌구",
        documents[0].page_content
    ]
)

In [26]:
print(len(embedding_vector), len(embedding_vector[0]), len(embedding_vector[1]))
print(embedding_vector[0][:10])

2 3072 3072
[0.014169108122587204, -0.022027326747775078, 0.009101607836782932, 0.009438703767955303, 0.030725523829460144, 0.0165895726531744, -0.026680365204811096, 0.011428126133978367, -0.014898562803864479, 0.01189232524484396]


In [11]:
%%time
from langchain_chroma import Chroma
# 데이터 처음 저장할 때
# database = Chroma.from_documents(
#     documents=documents, # chunk
#     embedding=embedding, # 임베딩 객체
#     collection_name="tax-collection", # 생략시 이름 랜덤
#     persist_directory='./chroma'      # 생략시 로컬DB에 저장 안 되어 프로그램 종료시 DB 제거됨
# )
# 이미 저장된 vector DB(store)를 사용할 때
database = Chroma(
    embedding_function=embedding,
    collection_name='tax-collection',
    persist_directory='./chroma'
)

CPU times: total: 875 ms
Wall time: 6.88 s


In [35]:
results = database._collection.get(include=['embeddings', 'documents','metadatas'])
print("데어터 수:",len(results['ids']))
print('문서 임베딩 차원 수 :', len(results['embeddings'][0]))
print("1번째 임베딩 샘플 :", results['embeddings'][1])
print("1번째 원본 :", results['documents'][1][:50])
print('1번쨰 metadatas :', results['metadatas'][1])

데어터 수: 180
문서 임베딩 차원 수 : 3072
1번째 임베딩 샘플 : [ 0.01991478 -0.01470464 -0.00057961 ...  0.0058937  -0.03365059
 -0.00657188]
1번째 원본 : . 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이 분배되는 
1번쨰 metadatas : {'source': './data/소득세법(법률)(제21065호)(20260102).docx'}


# 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [36]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieval_docs = database.similarity_search(query=query,
                                           k=2) # 기본 k값은 4

In [43]:
# retrieval_docs

In [45]:
# print("\n\n--\n\n".join([doc.page_content for doc in retrieval_docs]))
retrieval_doc = "\n\n--\n\n".join([doc.page_content for doc in retrieval_docs])

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성 -1

In [47]:
from langchain_openai import ChatOpenAI
load_dotenv()
llm=ChatOpenAI(model="gpt-4.1-nano")

In [48]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세 전문가입니다
- [context]를 참고하여 사용자의 질문에 답변해 주세요
[context]의 내용은 다음과 같아요
{retrieval_doc}
질문:{query}"""

In [50]:
ai_message = llm.invoke(prompt)

In [52]:
ai_message.usage_metadata

{'input_tokens': 2134,
 'output_tokens': 867,
 'total_tokens': 3001,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [54]:
print(ai_message.content)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 다음과 같은 절차를 따릅니다.

1. 과세표준 산정
2. 세율 적용
3. 표준공제 및 보험료 공제 등을 고려한 최종 세액 산출

**1. 과세표준 산정**
- 연봉이 50,000,000원일 경우, 기본공제(본인공제 150만원) 등을 차감하여 과세표준을 산출합니다.
- 일반적으로 근로소득공제는 다음과 같습니다.

| 연간 총소득 | 근로소득공제액(자율계산) |
|--------------|-------------------------|
| 5,000만원 이하 | 소득의 55% 또는 한도 내 최대(2000만원) 중 적은 금액 |
| 5,000만원 초과 | 한도 내 55% (이 경우 대략 27,500,000원) |

이 경우, 연봉이 5천만원이므로, 근로소득공제는 대략 2,750만원입니다.

**근로소득공제 = 27,500,000원**

즉, 과세표준 계산:
- 연봉: 50,000,000원
- 공제: 약 27,500,000원 (근로소득공제)
- 기본공제(본인): 1,500,000원

과세표준:
50,000,000 - 27,500,000 - 1,500,000 = **월 21,000,000원**

하지만, 상세 계산 시, 근로소득공제액은 정부의 정한 표에 따라 차등 적용됩니다. 일반적으로 연봉 5,000만원 기준, 근로소득공제는 약 2,750만원으로 예상됩니다.

**2. 세율 적용**
- 과세표준에 따른 세율은 다음과 같습니다. (2023년 기준)

| 과세표준 구간 | 세율 | 누진공제 |
|----------------|--------|-----------|
| 1,200만원 이하 | 6% | 0 |
| 1,200만원 초과 ~ 4,600만원 이하 | 15% | 108만원 |
| 4,600만원 초과 ~ 8,800만원 이하 | 24% | 522만원 |
| 8,800만원 초과 ~ 1억 5,000만원 이하 | 35% | 1,490만원 |
| 1억 5,000만원 초과 | 38% | 1,950만원 |

과세표준이 5,000

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성 -2

In [59]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
load_dotenv()
llm=ChatOpenAI(model="gpt-4.1-nano")
promptTemplate=ChatPromptTemplate([
    ("system", "당신은 최고의 한국 소득세 전문가입니다."),
    ("human", f"""다음 문맥을 참고하여 질문에 답변하세요.
    답을 모르면 모른다고 말하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 : """)
])
promptTemplate

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 최고의 한국 소득세 전문가입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 문맥을 참고하여 질문에 답변하세요.\n    답을 모르면 모른다고 말하세요.\n    최대 3문장으로 간결하게 답변하세요.\n    질문 : {question}\n    문맥 : {context}\n    답변 : '), additional_kwargs={})])

In [60]:
prompt = promptTemplate.invoke({
    'context':retrieval_doc, # retrival_docs보다 추천
    'question' : query
})

In [62]:
llm.invoke(prompt)

AIMessage(content='연봉 5천만원인 직장인의 소득세는 일반적인 과세 표준과 공제액에 따라 계산해야 하며, 세율에 따라 대략 7~15% 정도가 될 것으로 예상됩니다. 그러나 정확한 금액은 총급여액, 공제항목, 세액공제 여부에 따라 달라지므로 구체적인 계산이 필요합니다. 구체적인 계산을 위해서는 상세한 소득 및 공제 항목 정보가 필요합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 2156, 'total_tokens': 2261, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'id': 'chatcmpl-CvauJEe45sLZRrCcwELUEfR35B5Ek', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b9ba1-5cc9-7371-ab20-55103ca70577-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2156, 'output_tokens': 105, 'total_tokens': 2261, 'input_token_details': {'audio': 0, 'cache_read': 0

In [63]:
# 위의 예제를 langchain으로 답변생성
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(promptTemplate.invoke({
    'context':retrieval_doc,
    'question' : query
})))

'연봉 5천만원인 직장인의 소득세는 정확한 계산을 위해 근로소득공제와 기본공제, 세율 등 추가 정보가 필요합니다. 그러나 일반적으로 5천만원의 연봉에 대한 근로소득세는 약 600만원 내외가 예상됩니다. 자세한 세금액은 개인의 공제 조건에 따라 달라질 수 있습니다.'

# 6. langchain으로 답변 생성

In [64]:
# 위의 예제를 langchain으로 답변생성
rag_chain = promptTemplate | llm | output_parser
rag_chain.invoke({'context':retrieval_doc,'question':query})

'연봉 5천만원인 직장인의 소득세는 정확한 세율 계산이 필요하나, 대략적으로 10~15% 수준입니다. 이는 근로소득공제와 세율 구간에 따라 달라지며, 구체적 계산은 세액공제, 근로소득공제 및 지방소득세 등을 고려해야 합니다. 정확한 세액을 알고 싶으면 세무사와 상담하는 것을 추천드립니다.'

## langchain 전달
    smith.langchain.com 에서 key 생성 후 .env에 LANGCHAIN_API_KEY 추가

In [16]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 1. LLM과 임베딩 초기화
load_dotenv()
llm = ChatOpenAI(model="gpt-4.1-mini")
embedding = OpenAIEmbeddings(model="text-embedding-3-large")
# 2. vector store load
vectorstore = Chroma(
    embedding_function=embedding,
    collection_name="tax-collection",
    persist_directory="./chroma/"
)
# 3. Retriever 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)
# 4. 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세 전문가입니다.
다음 문맥을 참고하여 질문에 답하세요
답을 모르면 모른다고 답하세요
최대 3문장으로 간결하게 답변하세요.
질문:{{query}}
문맥:{{context}}
답변:"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 document를 텍스트로 변환하는 함수
def format_documents(documents):
    return "\n\n--\n\n".join([doc.page_content for doc in documents])

In [17]:
# 6. RAG 체인 구성(LCEL 방식)
from langchain_core.runnables import RunnablePassthrough # {"query":"~"}=>"~"
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문 그대로 전달
    }
    | prompt # prompt에 cdontext와 query 변수 주입
    | llm 
    | StrOutputParser()
)
# 7. 실행
query ="연봉 5천만원인 직장인의 소득세는 얼마인가요?"
rag_chain.invoke(query)

'연봉 5천만원인 직장인의 소득세는 개인별 공제항목, 세율구간, 4대 보험료 공제 등을 고려해야 정확히 산출됩니다. 단순 계산 시 근로소득공제를 적용한 과세표준에 대해 국세청의 누진세율(예: 6%~42%)을 적용하여 산출하며, 약 300만~400만원 내외로 예상됩니다. 정확한 금액은 연말정산 자료 및 추가 공제사항에 따라 달라지므로 상세한 계산이 필요합니다.'